In [47]:
from typing import Any, Dict
from pathlib import Path

import numpy as np

from SysSimX.core.base import CoSimComponent
from SysSimX.core.port import PortSpec, PortType
from SysSimX.core.events import EventLocator
from SysSimX.utilities.units import Quantity
from SysSimX.components.fmu_comp import FMUComponent

In [ ]:
class Pendulum(FMUComponent):
    """
    FMU-based pendulum component with rollback support.
    """
    def __init__(self, name, group = None):
        fmu_path = Path.cwd().parent / "test_data" / "FMUs" / "Pendulum_Cvode.fmu"
        super().__init__(name, fmu_path, group)

        contact_port_spec = PortSpec(
            name="contact",
            type=PortType.BOOL,
            direction="out"
        )
        self.output_specs.update({"contact": contact_port_spec})

    def snapshot_state(self):
        """Create a snapshot of the current component state."""
        state = super().get_state()
        state["t"] = self.t
        return state

    def restore_state(self, snapshot, t):
        """Restore component to a previous state from snapshot."""
        self.t = t
        q0 = snapshot['q']['value']
        omega0 = snapshot['omega']['value']
        self._instance.reset()
        self._instance.instantiate()
        self._instance.setupExperiment(startTime=t)
        self._instance.enterInitializationMode()
        self.set_parameters(**{'q0': q0, 'omega0': omega0})
        self._apply_parameters_starts()
        self._apply_input_starts()
        self._instance.exitInitializationMode()

        self._update_output_states(t)
        self._record_outputs(t)

    def _handle_events_internal(self, event_names, t):
        if "wall_hit" not in event_names:
            return
        restitution = 1
        output = self.get_outputs()
        q0 = output['q'].magnitude
        omega0 = -restitution * output['omega'].magnitude

        self._instance.reset()
        self._instance.instantiate()
        self._instance.setupExperiment(startTime=t)
        self._instance.enterInitializationMode()
        self.set_parameters(**{'q0': q0, 'omega0': omega0})
        self._apply_parameters_starts()
        self._apply_input_starts()
        self._instance.exitInitializationMode()

        self._update_output_states(t)
        # Update contact output port
        self.outputs['contact'].set(True, t=t)

        self._record_outputs(t)

pendulum = Pendulum(name="Pendulum")
pendulum.set_parameters(q0=0.3)

def event_indicator(comp: Pendulum) -> float:
    """Event indicator function for the pendulum component."""
    return comp.get_outputs()['q']

pendulum.add_event_indicator(name="wall_hit",
                             func=event_indicator,
                             direction=-1)

LOG_SOLVER        | info    | CVODE linear multistep method CV_BDF
LOG_SOLVER        | info    | CVODE maximum integration order CV_ITER_NEWTON
LOG_SOLVER        | info    | CVODE use equidistant time grid YES
LOG_SOLVER        | info    | CVODE Using relative error tolerance 1.000000e-06
LOG_SOLVER        | info    | CVODE Using dense internal linear solver SUNLinSol_Dense.
LOG_SOLVER        | info    | CVODE Use internal dense numeric jacobian method.
LOG_SOLVER        | info    | CVODE uses internal root finding method NO
LOG_SOLVER        | info    | CVODE maximum absolut step size 0
LOG_SOLVER        | info    | CVODE initial step size is set automatically
LOG_SOLVER        | info    | CVODE maximum integration order 5
LOG_SOLVER        | info    | CVODE maximum number of nonlinear convergence failures permitted during one step 10
LOG_SOLVER        | info    | CVODE BDF stability limit detection algorithm OFF


In [ ]:
class PID(FMUComponent):
    """
    FMU-based PID controller component.
    """
    def __init__(self, name, group = None):
        fmu_path = Path.cwd().parent / "test_data" / "FMUs" / "PID_Continuous.fmu"
        super().__init__(name, fmu_path, group)
    
    def _handle_events_internal(self, event_names, t):
        if "wall_hit" not in event_names:
            return
        self.set_inputs({'resetI': True}, t=t)
        self.do_step(t, 0)
        self.set_inputs({'resetI': False}, t=t)

    def snapshot_state(self):
        return self._instance.getFMUstate()
    
    def restore_state(self, snapshot, t):
        self.t = t
        self._instance.setFMUstate(snapshot)
        self._update_output_states(t)
        self._record_outputs(t)


({'I_out': 40.00000000000002, 'u': 1.0}, 2.0000000000000013)

({'I_out': 0.0, 'u': 1.0}, 0.0)

In [ ]:
from SysSimX.system.connection import Connection
from SysSimX.system.system import System


